Kütüphaneleri yüklüyoruz.

In [ ]:
import time
import torch
import pandas as pd
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

Modelleri indirmek için HuggingFace platformuna Token ile giriş yapıyoruz.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# 1. Adım: Kilitli kasadan (Secrets) şifremizi güvenle çekiyoruz
gizli_token = userdata.get('HF_TOKEN')

# 2. Adım: HuggingFace'e arka planda sessizce giriş yapıyoruz
login(gizli_token)

print("✅ HuggingFace'e otomatik giriş başarıyla yapıldı!")

✅ HuggingFace'e otomatik giriş başarıyla yapıldı!


model_id kısmına yüklemek istediğimiz modelin id'sini yazıyoruz.

In [ ]:
model_id = "google/gemma-2-9b-it" # Hızlı test edebilmek için küçük bir model seçtik
print(f"--- {model_id} Modeli İndiriliyor (Bu biraz sürebilir) ---")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)
print("✅ Model başarıyla yüklendi ve hazır!")

--- google/gemma-2-9b-it Modeli İndiriliyor (Bu biraz sürebilir) ---


config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

✅ Model başarıyla yüklendi ve hazır!


Modelden gelen cevapları değerlendirmek için Anlamsal benzerlik gibi metodlar kullanıyoruz. İsteyen buralardaki ölçüm mdetodlarını revize edebiir.

In [ ]:
print("Anlamsal benzerlik modeli yükleniyor...")
anlamsal_benzerlik_modeli = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

def cevap_dogru_mu(dogru_cevap, verilen_cevap,secenekler):
    ops = {0: "A", 1: "B", 2: "C", 3: "D"}
    verilen_cevap = verilen_cevap.upper().strip()

    if dogru_cevap == verilen_cevap:
        return True
    elif len(verilen_cevap) > 1 and verilen_cevap[1] in [" ", ":", ")", "=", "-", ".","\n"]:
        print("Şuan ikinci kısımda")

        if verilen_cevap[0] == dogru_cevap:
            return True
        else:
          print("Anlam uzayına girdi")
          encoded_cevap = anlamsal_benzerlik_modeli.encode([verilen_cevap])
          encoded_secenekler = anlamsal_benzerlik_modeli.encode(secenekler)
          benzerlik_listesi = anlamsal_benzerlik_modeli.similarity(encoded_cevap, encoded_secenekler).tolist()[0]
          en_yuksek_benzerlik_index = benzerlik_listesi.index(max(benzerlik_listesi))
          return ops[en_yuksek_benzerlik_index] == dogru_cevap

Anlamsal benzerlik modeli yükleniyor...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Veri setini yüklüyoruz.

In [ ]:
print("MMLU Veri seti indiriliyor...")
benchmark_id = "hf://datasets/Endezyar/siyer_benchmark/data/train-00000-of-00001.parquet"
mmlu_veri = pd.read_parquet(benchmark_id)
print("✅ Veri seti başarıyla yüklendi!"

MMLU Veri seti indiriliyor...


İndirdiğimiz modele bir fonksiyon ile erişmek istiyoruz.

In [ ]:
def sor(prompt):
  print("Model düşünüyor...")
  # 1. Soruyu modelin anlayacağı sayılara (token) çeviriyoruz
  inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

  # 2. Modeli çalıştırıp cevabı üretiyoruz
  with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=25,
          max_length=None, # Uyarıyı susturan mimari dokunuş
          temperature=0.01,
          do_sample=False
      )

  # 3. Modelin ürettiği sayıları tekrar bizim okuyabileceğimiz metne çeviriyoruz
  gelen_metin = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

  return gelen_metin

ops benim veri setimi düzenlendiğim formattır. Asıl olan prompt yazarken soruyu ve seçenekleri vermektir. Sonra alınan cevap cevap_dogru_mu fonksiyonuna gönderilir. Bu fonksiyon soru seçeneklerini de ister. Bundan dolayı burası eklenmiştir.

In [ ]:
def ops(metin):
  satirlar = metin.strip().split('\n')
  sik_satirlari = satirlar[1:]
  secenekler = []
  for satir in sik_satirlari:
      ikinci_kisim = satir.split(':')[1].strip()
      secenekler.append(ikinci_kisim)
  return secenekler

Burada tüm sorular verilir ve çıktılar değerlendirilir.

In [ ]:

baslama_zamani = time.time()
dogru_cevap_sayisi = 0
toplam_soru = len(mmlu_veri)
for i in range(toplam_soru):
  soru_metni = mmlu_veri.iloc[i]['soru'] + "\n"
  secenekler = ops(soru_metni)
  dogru_cevap =  mmlu_veri.iloc[i]['dogru_cevap']

  prompt = f"Sana soru ve seçenekleri veriyorum. İyi düşün. Doğru cevapların ödüllendirilecek. Sadece A, B, C ve D seçeneklerin var. İlk karakter olarak hangi seçeğin doğru olduğunu şık olarak yaz. Sonra nedenini açıkla. Bold Kullanma.\nSoru:\n{soru_metni}\nCevap:"

  gelen_metin = sor(prompt)
  sonuc = cevap_dogru_mu(dogru_cevap, gelen_metin, secenekler)

  if sonuc:
    dogru_cevap_sayisi += 1

  simdi = time.time()
  gecen_sure = round(simdi - baslama_zamani, 2)
  basari_orani = round((dogru_cevap_sayisi / (i + 1)) * 100, 2)
  print(f"\rSoru: {i+1}/{toplam_soru} | Doğru: {dogru_cevap_sayisi} | Başarı: %{basari_orani} | Süre: {gecen_sure}s", end="")

toplam_sure = round(time.time() - baslama_zamani, 2)
genel_basari = round((dogru_cevap_sayisi / toplam_soru) * 100, 2)

print(f"\n\n✅ {benchmark_id} Benchmark yapısı uygulandı!")
print(f"\n\n✅ {model_id} Testi Tamamlandı!")
print(f"Başarı Oranı: %{genel_basari} | Toplam Süre: {toplam_sure} saniye")

Model düşünüyor...
Soru: 1/155 | Doğru: 0 | Başarı: %0.0 | Süre: 1.47sModel düşünüyor...
Şuan ikinci kısımda
Soru: 2/155 | Doğru: 1 | Başarı: %50.0 | Süre: 3.02sModel düşünüyor...
Şuan ikinci kısımda
Anlam uzayına girdi
Soru: 3/155 | Doğru: 1 | Başarı: %33.33 | Süre: 4.62sModel düşünüyor...
Şuan ikinci kısımda
Soru: 4/155 | Doğru: 2 | Başarı: %50.0 | Süre: 6.18sModel düşünüyor...
Şuan ikinci kısımda
Soru: 5/155 | Doğru: 3 | Başarı: %60.0 | Süre: 7.74sModel düşünüyor...
Şuan ikinci kısımda
Anlam uzayına girdi
Soru: 6/155 | Doğru: 4 | Başarı: %66.67 | Süre: 9.32sModel düşünüyor...
Şuan ikinci kısımda
Soru: 7/155 | Doğru: 5 | Başarı: %71.43 | Süre: 10.91sModel düşünüyor...
Şuan ikinci kısımda
Soru: 8/155 | Doğru: 6 | Başarı: %75.0 | Süre: 12.47sModel düşünüyor...
Şuan ikinci kısımda
Soru: 9/155 | Doğru: 7 | Başarı: %77.78 | Süre: 14.01sModel düşünüyor...
Şuan ikinci kısımda
Anlam uzayına girdi
Soru: 10/155 | Doğru: 8 | Başarı: %80.0 | Süre: 15.61sModel düşünüyor...
Şuan ikinci kısımda
Sor

Eğer soruları toplu vermek yerine tek tek bakıp yapmak isterseniz burayı kullanabilirsiniz.

In [ ]:

i = 5

soru_metni = mmlu_veri.iloc[i]['soru'] + "\n"
secenekler = ops(soru_metni)
dogru_cevap =  mmlu_veri.iloc[i]['dogru_cevap']

prompt = f"Sana soru ve seçenekleri veriyorum. İyi düşün. Doğru cevapların ödüllendirilecek. Sadece A, B, C ve D seçeneklerin var. İlk karakter olarak hangi seçeğin doğru olduğunu şık olarak yaz. Sonra nedenini açıkla. Bold Kullanma. \nSoru:\n{soru_metni}\nCevap:"

print("Modele Gönderilecek Soru (Prompt) Şu Şekilde Oldu:\n")
print(prompt)
print(dogru_cevap)
print("\n" + "="*50)
gelen_metin = sor(prompt)
print(f"Modelin Cevabı: {gelen_metin}")
print("\n" + "="*50)

# Hakem fonksiyonumuzu çağırıp modelin cevabını kontrol ediyoruz
sonuc = cevap_dogru_mu(dogru_cevap, gelen_metin, secenekler)

print(f"Veri Setindeki Gerçek Cevap: {sonuc}")

if sonuc:
    print("✅ TEBRİKLER: Model soruyu DOĞRU bildi!")
else:
    print("❌ MAALESEF: Model soruyu YANLIŞ bildi!")

Modele Gönderilecek Soru (Prompt) Şu Şekilde Oldu:

Sana soru ve seçenekleri veriyorum. İyi düşün. Doğru cevapların ödüllendirilecek. Sadece A, B, C ve D seçeneklerin var. İlk karakter olarak hangi seçeğin doğru olduğunu şık olarak yaz. Sonra nedenini açıkla. Bold Kullanma. 
Soru:
İlk vahyin geldiği mübarek gece hangisidir?
A:İsra Gecesi
B:Mirac Gecesi
C:Kadir Gecesi
D:Berat Gecesi


Cevap:
C

Model düşünüyor...
Modelin Cevabı: 
B
Mirac Gecesi, Hz. Muhammed'in (s.a.v.) Mekke'den

Şuan ikinci kısımda
Anlam uzayına girdi
Veri Setindeki Gerçek Cevap: True
✅ TEBRİKLER: Model soruyu DOĞRU bildi!
